In [0]:
%sql
-- drops gold dimensional view and then creates it
drop view if exists gold_dim_customers;
create view gold_dim_customers as
select
  row_number() over (order by customer_id) as customer_key, -- surrogate key
  ci.customer_id,
  ci.customer_key as customer_number,
  ci.firstname,
  ci.lastname,
  la.country as country,
  case 
    when ci.gender != 'N/A' then ci.gender
    else coalesce(ca.gender, 'N/A')
  end as gender,
  ca.birthdate as birthdate,
  ci.marital_status,
  ci.customer_creation_date,
  ci.dlh_create_date
from workspace.`lakehouse-transformation-project`.silver_crm_cust_info as ci
left join workspace.`lakehouse-transformation-project`.silver_erp_cust_az_12 as ca
  on ci.customer_key = ca.cid
left join workspace.`lakehouse-transformation-project`.silver_erp_loc_a_101 as la
  on ci.customer_key = la.cid

In [0]:
%sql
-- drops the view and then recreates it; dimensional products
drop view if exists gold_dim_products;
create view gold_dim_products as
select 
  row_number() over (order by pn.prd_start_dt, pn.product_key) as product_key, -- surrogate
  pn.prd_id as product_id,
  pn.product_key as product_number,
  pn.prd_nm as product_name,
  pn.category_id,
  px.cat as category,
  px.subcat as subcategory,
  px.maintenance as maintenance,
  pn.product_cost,
  pn.product_line,
  pn.prd_start_dt as prod_start_date,
  current_timestamp() as dlh_create_date 
from workspace.`lakehouse-transformation-project`.silver_crm_prd_info as pn
left join workspace.`lakehouse-transformation-project`.silver_erp_px_cat_g_1_v_3 as px
  on pn.category_id = px.ID
where pn.prd_end_dt is null

In [0]:
%sql
-- drops the gold facts view and then recreates it
drop view if exists gold_fact_sales;
create view gold_fact_sales as 
select
  sd.order_number,
  pr.product_key,
  cu.customer_key,
  sd.order_date, 
  sd.ship_date,
  sd.due_date,
  sd.sales,
  sd.quantity,
  sd.price
from workspace.`lakehouse-transformation-project`.silver_crm_sales_details as sd
left join workspace.`lakehouse-transformation-project`.gold_dim_products as pr
  on sd.product_key = pr.product_number
left join workspace.`lakehouse-transformation-project`.gold_dim_customers as cu  
  on sd.customer_id = cu.customer_id